# Unsupervised Learning on Country Data
<img src="images/country.png" height=60% width=60%>

# HELP International

HELP International is an international humanitarian NGO that is committed to fighting poverty and providing the people of backward countries with basic amenities and relief during the time of disasters and natural calamities.

## Problem Statement

HELP International have been able to raise around $ 10 million. Now the CEO of the NGO needs to decide how to use this money strategically and effectively. So, CEO has to make decision to choose the countries that are in the direst need of aid. Hence, your Job as a Data scientist is to categorise the countries using some socio-economic and health factors that determine the overall development of the country. Then you need to suggest the countries which the CEO needs to focus on the most.

## Objective

To categorise the countries using socio-economic and health factors that determine the overall development of the country.

# Preview Data

In [51]:
# import pandas
import pandas as pd

# read data
country = pd.read_csv('Data Zip/archive/Country-data.csv')

# data info
country.info()

<class 'pandas.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    str    
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 13.2 KB


In [52]:
# first 5 rows
country.head()

,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


# Description of Columns:
* **country** - Name of the country
* **child_mort** - Death of children under 5 years of age per 1000 live births
* **exports** - Exports of goods and services per capita. Given as %age of the GDP per capita
* **health** - Total health spending per capita. Given as %age of GDP per capita
* **imports** - Imports of goods and services per capita. Given as %age of the GDP per capita
* **Income** - Net income per person
* **Inflation** - The measurement of the annual growth rate of the Total GDP
* **life_expec** - The average number of years a new born child would live if the current mortality patterns are to remain the same
* **total_fer** - The number of children that would be born to each woman if the current age-fertility rates remain the same.
* **gdpp** - The GDP per capita. Calculated as the Total GDP divided by the total population.


In [ ]:
# import graphing tools
import matplotlib.pyplot as plt
import seaborn as sns

# pair plot
sns.pairplot(country.select_dtypes(exclude=['object']))
plt.show()

# Descriptive Statistics

In [ ]:
country.describe(percentiles=[0.10, .25, .50, .75, .90 ,.95, .99]).T

## Correlation

In [ ]:
# import numpy
import numpy as np
numerical_country = country.select_dtypes(exclude=["object"])

matrix = np.triu(numerical_country.corr(method='pearson'))
sns.heatmap(numerical_country.corr(method='pearson'),annot=True, vmin=-1, vmax=1, center= 0, mask=matrix, cmap='coolwarm', cbar=False, fmt='.1g')
plt.show()
numerical_country.corr()

# Top 25

In [ ]:
# function for table & graph
def top_25(feature):
    # top 25
    max_data = country.sort_values(feature, ascending=False).head(25)

    plt.figure(figsize=(6, 8))  
    sns.barplot(y='country', x=feature, data=max_data, palette='Reds_r', hue='country')
    plt.title(f'25 Countries with Highest {feature}', fontsize=14, pad=15)
    plt.xlabel(feature.replace('_', ' ').title())
    plt.ylabel('Country')
    plt.show()

    # Display only relevant columns
    display(max_data[['country', feature]])

    # low 25
    min_data = country.sort_values(feature, ascending=True).head(25)

    plt.figure(figsize=(6, 8))
    sns.barplot(y='country', x=feature, data=min_data, palette='Blues', hue='country')
    plt.title(f'25 Countries with Lowest {feature}', fontsize=14, pad=15)
    plt.xlabel(feature.replace('_', ' ').title())
    plt.ylabel('Country')
    plt.show()

    display(min_data[['country', feature]])

## Child Mortality 

In [ ]:
top_25('child_mort')

## Exports

In [ ]:
top_25('exports')

## Health

In [ ]:
top_25('health')

## Imports

In [ ]:
top_25('imports')

## Income

In [ ]:
top_25('income')

## Inflation

In [ ]:
top_25('inflation')

## Life Expectancy

In [ ]:
top_25('life_expec')

## Total Fertility

In [ ]:
top_25('total_fer')

## GDP per Capita

In [ ]:
top_25('gdpp')

# Unsupervised Learning

In [ ]:
# import Kmeans, Scaler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# preprocessing pipeline (scale)
preprocessing_pipeline_steps = []
preprocessing_pipeline_steps.append(('scaler', StandardScaler()))
preprocessing_pipeline = Pipeline(steps=preprocessing_pipeline_steps)

# scaled data
country_scaled = preprocessing_pipeline.fit_transform(country.drop('country', axis='columns'))

## Kmeans

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# calculate metrics
inertia = []
sil_scores = []
k_range = range(2, 8)

for n_clusters in k_range:
    kmeans = KMeans(n_clusters=n_clusters, random_state=2025, max_iter=1_000, n_init='auto')
    kmeans.fit(country_scaled)
    
    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(country_scaled, kmeans.labels_))

# combine data into a single clean DataFrame
metrics_df = pd.DataFrame({
    'n_clusters': k_range,
    'inertia': inertia,
    'silhouette_score': sil_scores
})

# plot side-by-side using Subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# plot 1: Elbow Method (Inertia)
sns.lineplot(ax=axes[0], x='n_clusters', y='inertia', data=metrics_df, marker='o', color='b')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Average Within-Cluster Squared Distances')
axes[0].set_xticks(k_range)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot 2: Silhouette Scores
sns.lineplot(ax=axes[1], x='n_clusters', y='silhouette_score', data=metrics_df, marker='o', color='orange')
axes[1].set_title('Silhouette Analysis')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(k_range)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=2025, max_iter=1_000).fit(country_scaled)
kmeans_labels = kmeans.labels_ + 1

## Hierarchial Clustering

In [ ]:
# import
from sklearn.cluster import AgglomerativeClustering

linkages = ['ward', 'single', 'complete', 'average']
for link in linkages:
    sil_scores = []
    for i in range(2,6):
        cluster = AgglomerativeClustering(n_clusters=i, linkage=link)
        hier_model = cluster.fit(country_scaled)
        labels = hier_model.labels_
        sil_scores.append(silhouette_score(country_scaled, labels))

    sns.lineplot(x=range(2,6), y=sil_scores, markers=True, label=link)
    sns.scatterplot(x=range(2,6), y=sil_scores, markers=True)
    plt.title('Hierarchial Clustering')
    plt.xlabel('number of clusters')
    plt.ylabel('silhouette score')
plt.show()

In [ ]:
cluster = AgglomerativeClustering(n_clusters=3, linkage='ward')
hier_model = cluster.fit(country_scaled)
hier_labels = hier_model.labels_ + 1

# TSNE

In [ ]:
from sklearn.manifold import TSNE
country_tsne = pd.DataFrame(TSNE(n_components=2).fit_transform(country_scaled), columns=['x','y'])

# kmeans
sns.scatterplot(x='x',y='y', data=country_tsne, hue=kmeans_labels, palette='Set1')
plt.title('KMeans Clustering')
plt.show()

# hier
sns.scatterplot(x='x',y='y', data=country_tsne, hue=hier_labels, palette='Set1')
plt.title('Hierarchial Clustering')
plt.show()

# Analysing Clusters
Based on the TSNE graphs of the clusters, 3 clusters looks the best. Using Kmeans Clustering labels because the graph looks best.

In [ ]:
# labels used
labels_used = kmeans_labels
# concat labels to data set
country_with_labels = pd.concat([country, pd.Series(labels_used)], axis='columns')
country_with_labels.rename({0:'cluster'}, axis='columns', inplace=True)

## Summary of Clusters

### Cluster 1

In [ ]:
# Cluster 1
display(country_with_labels[country_with_labels.cluster==1].sort_values('gdpp',ascending=False))
country_with_labels[country_with_labels.cluster==1].describe()

### Cluster 2

In [ ]:
# Cluster 2
display(country_with_labels[country_with_labels.cluster==2].sort_values('gdpp',ascending=False))
country_with_labels[country_with_labels.cluster==2].describe()

### Cluster 3

In [ ]:
# Cluster 3
display(country_with_labels[country_with_labels.cluster==3].sort_values('gdpp',ascending=False))
country_with_labels[country_with_labels.cluster==3].describe()

# Cluster Comparison

In [ ]:
def anova_stat(feature):
    df1=country_with_labels[country_with_labels.cluster==1][feature]
    df2=country_with_labels[country_with_labels.cluster==2][feature]
    df3=country_with_labels[country_with_labels.cluster==3][feature]
    statistic, p_value = stats.f_oneway(df1, df2,df3)
    return np.round(p_value, 5)
def comp_cluster(feature):
    temp_data = country_with_labels.groupby('cluster', as_index=False)[feature].mean()
    sns.barplot(x='cluster', y=feature, data=temp_data)
    sup_title = 'Cluster Comparison of ' + feature
    title= 'Anova p-value: ' + str(anova_stat(feature))
    plt.suptitle(sup_title)
    plt.title(title)
    plt.show()
    display(temp_data)
import scipy.stats as stats


In [ ]:
features=['child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
for feature in features:
    comp_cluster(feature)

# Recommendation
Based on the clusters, Cluster #2 seems to be in dire need of aid. It experiences very low GDP, high child mortality, very high fertility, and low life expectancy.

In [ ]:
country_with_labels[country_with_labels.cluster==2]